# Swaption out-of-sample test — step 1: bond prices

**Goal of the full study.** Reprice the ATM swaption grid (never used in calibration) with the
calibrated caplet parameters. A swaption needs the discount and projection bonds $P(T_e,T_i)$ *at the
option expiry*, conditional on the model state there. This notebook builds and validates that
ingredient; the swaption pricer is step 2.

**Model** (as in the paper, `dm_sim_stream.py`): $x$ is OU with variance $V = wV_1+(1-w)V_2$, the
$V_i$ are CIR, and the leverage correlates $dW^x$ with $dW^{v_1}$ only:

$$\mathrm{d}\langle x, V_1\rangle_t = \rho\,\varepsilon\sqrt{V V_1}\,\mathrm{d}t .$$

That cross term is **not affine** unless $w=1$, so $E_t[e^{-\int_t^T x}]$ is not exactly
exponential-affine. We use the ansatz $\exp(A + Bx + C_1V_1 + C_2V_2)$ with the
**homogeneous linearisation** $\sqrt{V V_1}\approx a_1V_1 + a_2V_2$, tangent at the long-run ratio
$r=\theta_2/\theta_1$ (exact when $V_2/V_1=r$, and for any state when $w=1$). Riccati system
($\tau=T-t$, all zero at $\tau=0$):

$$B = -\frac{1-e^{-\lambda\tau}}{\lambda},\qquad
C_1' = \tfrac12 wB^2 - \kappa_1C_1 + \tfrac12\varepsilon^2C_1^2 + \rho\varepsilon a_1BC_1,$$
$$C_2' = \tfrac12(1-w)B^2 - \kappa_2C_2 + \tfrac12\varepsilon^2C_2^2 + \rho\varepsilon a_2BC_1,\qquad
A' = \kappa_1\theta_1C_1 + \kappa_2\theta_2C_2,$$

(derivatives in $\tau$). The basis factor $k$ is an independent Gaussian HW factor and has the closed
form $\exp(A_k + B_k k)$.

**Validation here:** unconditional $E[e^{-\int_0^T x}]$ from the Riccati solution vs our Monte Carlo
simulator (the same engine the calibration uses), reported as a yield difference in bp. We also show
the $\rho=0$ solution, to measure how much the leverage term matters at all.

*CPU only, a few minutes. No GPU, nothing written outside this notebook.*

In [ ]:
import os, sys, math, numpy as np, pandas as pd, torch
DM = os.path.abspath(os.path.join(os.getcwd(), ".."))          # .../dm_calibration
sys.path.insert(0, DM); sys.path.insert(0, os.path.join(DM, "..", "scripts"))
from dm_calibrate import make_dm_space, constrain, load_curve_vec
from dm_sim_stream import simulate_price_streaming
from simulation import theta_to_vec, rho_to_vec
torch.set_num_threads(8); dev = torch.device("cpu")

CCY, TAG = "eur", "term_fc"                      # the headline EUR fit
ck = torch.load(os.path.join(DM, "runs", "calibration_weights", f"{CCY}_HWH_{TAG}.pt"), map_location="cpu")
space = make_dm_space(dev, kf_lo=ck.get("kf_lo"))
th1, th2, kf, ks, wf, eps, lam, gam, xi, r1, r5, r10, v0 = constrain("HWH", space["unpack"](ck["p"]))
print(f"kf={kf:.3f} ks={ks:.3f} w={wf:.3f} eps={eps:.3f} lam={lam:.3f} gam={gam:.3f} "
      f"xi={xi:.2e} rho=({r1:.2f},{r5:.2f},{r10:.2f}) v0={v0:.1e}")

In [ ]:
# daily grid to 10.5Y, theta(t) and rho(t) exactly as the simulator builds them
dt = 1 / 252; T_max = 10.5
n = int(math.ceil(T_max / dt)); tl = torch.arange(n + 1, dtype=torch.float64).mul(dt).float()
nodes, rho_nodes = space["theta_nodes"], space["rho_nodes_t"]
TH1 = theta_to_vec(th1, nodes, tl).double().numpy()
TH2 = theta_to_vec(th2, nodes, tl).double().numpy()
RHO = rho_to_vec(torch.tensor([r1, r5, r10]), rho_nodes, tl).double().numpy()

def riccati(t0, T, x0=0.0, V1=None, V2=None, leverage=True):
    # E_{t0}[exp(-int_{t0}^{T} x)] from state (x0, V1, V2); RK4 backward in tau on the daily grid
    i0, i1 = int(round(t0 / dt)), int(round(T / dt))
    V1 = v0 if V1 is None else V1; V2 = v0 if V2 is None else V2
    A = C1 = C2 = 0.0
    def rhs(tau, C1, C2, i):
        B = -(1 - math.exp(-lam * tau)) / lam
        r = TH2[i] / max(TH1[i], 1e-30); f = math.sqrt(wf + (1 - wf) * r)
        a1, a2 = (2 * wf + (1 - wf) * r) / (2 * f), (1 - wf) / (2 * f)
        rho = RHO[i] if leverage else 0.0
        d1 = 0.5 * wf * B**2 - kf * C1 + 0.5 * eps**2 * C1**2 + rho * eps * a1 * B * C1
        d2 = 0.5 * (1 - wf) * B**2 - ks * C2 + 0.5 * eps**2 * C2**2 + rho * eps * a2 * B * C1
        dA = kf * TH1[i] * C1 + ks * TH2[i] * C2
        return dA, d1, d2
    for j, i in enumerate(range(i1 - 1, i0 - 1, -1)):   # step tau from j*dt to (j+1)*dt, at calendar index i
        tau = j * dt
        k1 = rhs(tau, C1, C2, i)
        k2 = rhs(tau + dt / 2, C1 + dt / 2 * k1[1], C2 + dt / 2 * k1[2], i)
        k3 = rhs(tau + dt / 2, C1 + dt / 2 * k2[1], C2 + dt / 2 * k2[2], i)
        k4 = rhs(tau + dt, C1 + dt * k3[1], C2 + dt * k3[2], i)
        A  += dt / 6 * (k1[0] + 2 * k2[0] + 2 * k3[0] + k4[0])
        C1 += dt / 6 * (k1[1] + 2 * k2[1] + 2 * k3[1] + k4[1])
        C2 += dt / 6 * (k1[2] + 2 * k2[2] + 2 * k3[2] + k4[2])
    B = -(1 - math.exp(-lam * (T - t0))) / lam
    return math.exp(A + B * x0 + C1 * V1 + C2 * V2)

print("E[exp(-int_0^5 x)]  linearised:", riccati(0, 5.0), "  rho=0:", riccati(0, 5.0, leverage=False))

In [ ]:
# Monte Carlo: D(T) = E[exp(-int (f_ois + x))] from the calibration engine (no curve shift),
# so E[exp(-int x)] = D(T) / P_grid(T). Dummy caplets give the payment dates we want.
f_ois = torch.tensor(load_curve_vec(CCY, "ois", tl), dtype=torch.float32)
f_key = torch.tensor(load_curve_vec(CCY, "key", tl), dtype=torch.float32)
mats = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0]
i_pay = torch.tensor([int(round(T / dt)) for T in mats]); i_fix = i_pay - 1
K = torch.full((len(mats),), 1.0); tau1 = torch.tensor(dt)
cumf = np.concatenate([[0.0], np.cumsum(f_ois.double().numpy()[:-1] * dt)])

def mc_bond(v_init, n_paths=20000, seeds=(1, 2, 3)):
    out = []
    for sd in seeds:
        _, _, D = simulate_price_streaming(
            n_paths, tl, theta_to_vec(th1, nodes, tl), theta_to_vec(th2, nodes, tl), eps, v_init,
            kf, ks, wf, lam, gam, xi, f_key, f_ois, rho_to_vec(torch.tensor([r1, r5, r10]), rho_nodes, tl),
            i_fix, i_pay, tau1, K, dev, seed=sd)
        out.append(D.double().numpy() / np.exp(-cumf[i_pay.numpy()]))
    out = np.array(out)
    return out.mean(0), out.std(0, ddof=1) / math.sqrt(len(seeds))

rows = []
for label, vi in [("V(0)=v0 (calibrated)", v0), ("V(0)=1e-4 (~100bp vol)", 1e-4)]:
    m, se = mc_bond(vi)
    for T, mm, ss in zip(mats, m, se):
        lin, r0 = riccati(0, T, V1=vi, V2=vi), riccati(0, T, V1=vi, V2=vi, leverage=False)
        y = lambda e: -math.log(e) / T * 1e4                      # convexity yield, bp
        rows.append(dict(start=label, T=T, MC_bp=y(mm), MC_se_bp=ss / mm / T * 1e4,
                         linearised_minus_MC=y(lin) - y(mm), rho0_minus_MC=y(r0) - y(mm)))
tab = pd.DataFrame(rows).round(3); tab

**How to read the table.** `MC_bp` is the convexity yield $-\ln E[e^{-\int_0^T x}]/T$ in bp
(negative: the factor makes bonds dearer, which is the gap the curve shift $\varphi$ removes).
The last two columns are the Riccati error against Monte Carlo, in bp of yield.

*Pass criterion:* `linearised_minus_MC` within about 2 MC standard errors at every maturity, from both
starting states. `rho0_minus_MC` shows how much the leverage term contributes; if it is also tiny,
the bond prices are insensitive to the approximation altogether.

**Next (step 2, after review):** conditional bond prices at expiry from sampled states, checked by
nested MC; then the swaption pricer, checked against Henrard's exact Gaussian swaption formula
(Thm 5.3) at $\varepsilon=0$; then the ATM swaption grid vs market (`data/<ccy>/swaption_atm.csv`;
Bloomberg 1Y×5Y USD ATM = 111.52bp as a sanity point).